# Retail Sales Analysis – Advanced Business Insights

This notebook contains 10 **new business-focused questions** for team-leader reporting.
It uses the same eight CSV datasets as the original project and is divided into separate Google Colab sections.

**Required files:** `orders.csv`, `order_items.csv`, `products.csv`, `customers.csv`, `employees.csv`, `stores.csv`, `returns.csv`, `targets.csv`.

## Section 1: Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

## Section 2: Upload the eight CSV files

In [ ]:
from google.colab import files

print("Open the data folder and select all 8 CSV files.")
uploaded = files.upload()

DATA_DIR = Path("/content")
required_files = [
    "orders.csv", "order_items.csv", "products.csv", "customers.csv",
    "employees.csv", "stores.csv", "returns.csv", "targets.csv"
]
missing_files = [f for f in required_files if not (DATA_DIR / f).exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing files: " + ", ".join(missing_files) +
        ". Upload all eight CSV files from the data folder."
    )
print("All eight CSV files are ready.")

## Section 3: Load datasets and inspect available columns

In [ ]:
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
customers = pd.read_csv(DATA_DIR / "customers.csv")
employees = pd.read_csv(DATA_DIR / "employees.csv")
stores = pd.read_csv(DATA_DIR / "stores.csv")
returns = pd.read_csv(DATA_DIR / "returns.csv")
targets = pd.read_csv(DATA_DIR / "targets.csv")

for df in [orders, order_items, products, customers, employees, stores, returns, targets]:
    df.columns = df.columns.str.strip()

if "OrderDate" in orders.columns:
    orders["OrderDate"] = pd.to_datetime(orders["OrderDate"], errors="coerce")
if "Month" in targets.columns:
    targets["Month"] = pd.to_datetime(targets["Month"], errors="coerce")

for name, df in {
    "orders": orders, "order_items": order_items, "products": products,
    "customers": customers, "employees": employees, "stores": stores,
    "returns": returns, "targets": targets
}.items():
    print(f"\n{name}: {df.shape}")
    print(list(df.columns))

## Section 4: Prepare combined sales data

In [ ]:
sales_data = order_items.merge(orders, on="OrderID", how="left")
product_cols = [c for c in ["ProductID", "ProductName", "Category", "Price", "Cost"] if c in products.columns]
sales_data = sales_data.merge(products[product_cols], on="ProductID", how="left")

for col in ["Quantity", "Discount", "Price", "Cost"]:
    if col in sales_data.columns:
        sales_data[col] = pd.to_numeric(sales_data[col], errors="coerce").fillna(0)

if "Price" not in sales_data.columns or "Quantity" not in sales_data.columns:
    raise KeyError("The product/order item data must contain Price and Quantity fields.")

discount = sales_data["Discount"] if "Discount" in sales_data.columns else 0
cost = sales_data["Cost"] if "Cost" in sales_data.columns else 0
sales_data["Sales"] = sales_data["Price"] * sales_data["Quantity"] * (1 - discount)
sales_data["Profit"] = sales_data["Sales"] - cost * sales_data["Quantity"]

if "OrderDate" in sales_data.columns:
    sales_data["OrderDate"] = pd.to_datetime(sales_data["OrderDate"], errors="coerce")
    sales_data["Month"] = sales_data["OrderDate"].dt.to_period("M").dt.to_timestamp()

print("Combined sales data:", sales_data.shape)
display(sales_data.head())

## Question 1: Which stores are performing below average sales?

**Business purpose:** Identify stores that may need operational support.
**Graph:** Store sales compared with average store sales (bar chart).

In [ ]:
if "StoreID" not in sales_data.columns:
    print("Cannot calculate this question: StoreID is not available in the combined sales data.")
else:
    store_sales = sales_data.groupby("StoreID")["Sales"].sum().sort_values()
    avg_store_sales = store_sales.mean()
    colors = ["tomato" if x < avg_store_sales else "steelblue" for x in store_sales.values]
    store_sales.plot(kind="bar", color=colors)
    plt.axhline(avg_store_sales, color="black", linestyle="--", label=f"Average: {avg_store_sales:,.0f}")
    plt.title("Store Sales Compared with Average")
    plt.xlabel("Store ID"); plt.ylabel("Total Sales"); plt.legend()
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
    print("Stores below average:", list(store_sales[store_sales < avg_store_sales].index))

## Question 2: Which products have high sales but low profit margins?

**Business purpose:** Find products that sell well but may need cost or pricing review.
**Graph:** Sales versus profit margin (scatter plot).

In [ ]:
product_perf = sales_data.groupby("ProductName", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
product_perf["ProfitMarginPct"] = np.where(product_perf["Sales"] > 0, product_perf["Profit"] / product_perf["Sales"] * 100, 0)
sales_cutoff = product_perf["Sales"].quantile(0.75)
margin_cutoff = product_perf["ProfitMarginPct"].median()
product_perf["Flag"] = np.where((product_perf["Sales"] >= sales_cutoff) & (product_perf["ProfitMarginPct"] <= margin_cutoff), "High sales / lower margin", "Other")
sns.scatterplot(data=product_perf, x="Sales", y="ProfitMarginPct", hue="Flag")
plt.title("Product Sales vs Profit Margin")
plt.xlabel("Total Sales"); plt.ylabel("Profit Margin (%)"); plt.tight_layout(); plt.show()
display(product_perf.sort_values(["Sales", "ProfitMarginPct"], ascending=[False, True]).head(10))

## Question 3: Which customers contribute the most revenue?

**Business purpose:** Identify high-value customers for retention and loyalty activities.
**Graph:** Top 10 customers by revenue (horizontal bar chart).

In [ ]:
if "CustomerID" not in sales_data.columns:
    print("CustomerID is not available in the combined sales data.")
else:
    customer_sales = sales_data.groupby("CustomerID")["Sales"].sum().nlargest(10).sort_values()
    customer_sales.plot(kind="barh")
    plt.title("Top 10 Customers by Revenue")
    plt.xlabel("Total Revenue"); plt.ylabel("Customer ID")
    plt.tight_layout(); plt.show()
    print("Top customers' share of revenue:", f"{customer_sales.sum() / sales_data['Sales'].sum() * 100:.1f}%")

## Question 4: In which months do returns have the greatest impact on sales?

**Business purpose:** Monitor return patterns and their potential revenue impact.
**Graph:** Monthly sales and number of returns (two-axis chart).

In [ ]:
if "ReturnDate" in returns.columns:
    returns["ReturnDate"] = pd.to_datetime(returns["ReturnDate"], errors="coerce")
    returns["Month"] = returns["ReturnDate"].dt.to_period("M").dt.to_timestamp()
elif "OrderDate" in returns.columns:
    returns["OrderDate"] = pd.to_datetime(returns["OrderDate"], errors="coerce")
    returns["Month"] = returns["OrderDate"].dt.to_period("M").dt.to_timestamp()
else:
    returns["Month"] = pd.NaT

if "Month" in sales_data.columns and returns["Month"].notna().any():
    monthly_sales = sales_data.groupby("Month")["Sales"].sum()
    monthly_returns = returns.dropna(subset=["Month"]).groupby("Month").size()
    monthly = pd.concat([monthly_sales.rename("Sales"), monthly_returns.rename("Returns")], axis=1).fillna(0)
    ax = monthly["Sales"].plot(kind="bar", alpha=0.75, label="Sales")
    ax.set_ylabel("Sales")
    ax2 = ax.twinx()
    monthly["Returns"].plot(ax=ax2, color="darkorange", marker="o", label="Returns")
    ax2.set_ylabel("Number of returns")
    plt.title("Monthly Sales and Returns")
    plt.tight_layout(); plt.show()
    display(monthly)
else:
    print("A usable return date and sales month are required for this comparison.")

## Question 5: Which employees generate the highest sales?

**Business purpose:** Compare employee sales contribution and identify strong performers.
**Graph:** Top 10 employees by sales (bar chart).

In [ ]:
employee_col = next((c for c in ["EmployeeID", "SalespersonID"] if c in sales_data.columns), None)
if employee_col:
    employee_sales = sales_data.groupby(employee_col)["Sales"].sum().nlargest(10).sort_values()
    employee_sales.plot(kind="barh")
    plt.title("Top 10 Employees by Sales")
    plt.xlabel("Total Sales"); plt.ylabel("Employee")
    plt.tight_layout(); plt.show()
else:
    print("No EmployeeID or SalespersonID field was found in the combined sales data.")

## Question 6: Which stores have the highest product return rates?

**Business purpose:** Detect stores that may need improvements in product handling or customer service.
**Graph:** Return rate by store (bar chart).

In [ ]:
if "StoreID" in orders.columns and "OrderID" in orders.columns and "OrderID" in returns.columns:
    store_order_counts = orders.groupby("StoreID")["OrderID"].nunique()
    returned_order_counts = returns.merge(orders[["OrderID", "StoreID"]].drop_duplicates(), on="OrderID", how="left").groupby("StoreID")["OrderID"].nunique()
    return_rate = (returned_order_counts / store_order_counts * 100).fillna(0).sort_values(ascending=False)
    return_rate.plot(kind="bar")
    plt.title("Return Rate by Store")
    plt.xlabel("Store ID"); plt.ylabel("Returned orders (%)")
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
    display(return_rate.rename("ReturnRatePct").to_frame())
else:
    print("StoreID and OrderID fields are needed in orders and returns to calculate store return rates.")

## Question 7: How do discounts relate to profit margin?

**Business purpose:** Review whether larger discounts are associated with lower profitability.
**Graph:** Discount percentage versus profit margin (scatter plot).

In [ ]:
if "Discount" in sales_data.columns:
    discount_view = sales_data.copy()
    discount_view["DiscountPct"] = discount_view["Discount"] * 100
    discount_view["ProfitMarginPct"] = np.where(discount_view["Sales"] > 0, discount_view["Profit"] / discount_view["Sales"] * 100, 0)
    sns.scatterplot(data=discount_view, x="DiscountPct", y="ProfitMarginPct", alpha=0.6)
    plt.title("Discount Level vs Profit Margin")
    plt.xlabel("Discount (%)"); plt.ylabel("Profit Margin (%)")
    plt.tight_layout(); plt.show()
    print("Correlation:", round(discount_view[["DiscountPct", "ProfitMarginPct"]].corr().iloc[0, 1], 3))
else:
    print("Discount field is not available in the combined sales data.")

## Question 8: Which products have low sales and may need promotion?

**Business purpose:** Identify slow-moving products for promotion, assortment review, or stock planning.
**Graph:** 10 lowest-selling products (horizontal bar chart).

In [ ]:
product_sales = sales_data.groupby("ProductName")["Sales"].sum().sort_values().head(10)
product_sales.plot(kind="barh")
plt.title("10 Lowest-Selling Products")
plt.xlabel("Total Sales"); plt.ylabel("Product")
plt.tight_layout(); plt.show()
display(product_sales.to_frame("TotalSales"))

## Question 9: Which customer segments generate the highest average order value?

**Business purpose:** Understand which customer segments place higher-value orders.
**Graph:** Average order value by customer segment (bar chart).

In [ ]:
if "CustomerID" in orders.columns and "CustomerID" in customers.columns and "OrderID" in orders.columns:
    order_values = sales_data.groupby("OrderID")["Sales"].sum().reset_index()
    order_customer = orders[["OrderID", "CustomerID"]].drop_duplicates().merge(order_values, on="OrderID", how="left")
    segment_col = next((c for c in ["CustomerSegment", "Segment"] if c in customers.columns), None)
    if segment_col:
        order_customer = order_customer.merge(customers[["CustomerID", segment_col]], on="CustomerID", how="left")
        segment_aov = order_customer.groupby(segment_col)["Sales"].mean().sort_values(ascending=False)
        segment_aov.plot(kind="bar")
        plt.title("Average Order Value by Customer Segment")
        plt.xlabel("Customer Segment"); plt.ylabel("Average Order Value")
        plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()
        display(segment_aov.to_frame("AverageOrderValue"))
    else:
        print("Customer segment field not found in customers.")
else:
    print("CustomerID and OrderID fields are needed to calculate segment average order value.")

## Question 10: Which store and product-category combinations generate the most profit?

**Business purpose:** Identify the strongest store-category combinations and where improvement may be needed.
**Graph:** Store versus category profit heatmap.

In [ ]:
if "StoreID" in sales_data.columns and "Category" in sales_data.columns:
    store_category_profit = sales_data.pivot_table(index="StoreID", columns="Category", values="Profit", aggfunc="sum", fill_value=0)
    plt.figure(figsize=(12, 7))
    sns.heatmap(store_category_profit, annot=True, fmt=".0f", cmap="YlGnBu")
    plt.title("Profit by Store and Product Category")
    plt.xlabel("Product Category"); plt.ylabel("Store ID")
    plt.tight_layout(); plt.show()
    display(store_category_profit)
else:
    print("StoreID and Category fields are required for this analysis.")

## Section 15: Business KPI summary

In [ ]:
print(f"Total Sales: {sales_data['Sales'].sum():,.2f}")
print(f"Total Profit: {sales_data['Profit'].sum():,.2f}")
print(f"Total Orders: {orders['OrderID'].nunique():,}")
print(f"Total Customers: {customers['CustomerID'].nunique():,}")
print(f"Total Returns: {returns['ReturnID'].nunique():,}" if "ReturnID" in returns.columns else f"Total Returns: {len(returns):,}")